**Human Movement Intelligence | Presented by: Reza Saadatyar (2026)**

**09 | Full-Development Grouped Cross-Validation**

![Stage 09 scientific workflow](../../images/09_workflow.svg)

**Objective:** Compare three classical classifiers across five participant-disjoint Development folds and examine fold-level and paired participant-level performance.

**Dataset:** All 29 HARTH v2.0 Development participants, using the time-safe, 30-feature window cohort defined in Notebook 08.

**Notebook Scope:** Train and evaluate models within Development only; compare Random Forest and Logistic Regression on the same held-out participants. S023 and S032 remain excluded from this Notebook. Earlier exploratory inspection of those reserved recordings means this is not evidence of strict blind final Test evaluation.

**00. Colab Repository Setup**

In [ ]:
# ======================== 00. Colab Repository Setup =========================

# ------------------------- Repository Configuration --------------------------
from pathlib import Path

repo_url = "https://github.com/RezaSaadatyar/human-movement-intelligence.git"
repo_path = Path("/content") / repo_url.split("/")[-1].removesuffix(".git")

# ------------------------ Clone Repository if Missing ------------------------
%cd /content

if not repo_path.exists():
    !git clone -q {repo_url}

# ------------------------------ Open Repository ------------------------------
%cd {repo_path}


**01. Imports and Environment**

In [2]:
# ======================== 01. Imports and Environment ========================

# ---------------------- Study Environment Configuration ----------------------
from python_modules.core.colab_setup import setup_colab
root = setup_colab("wearable_activity_recognition")

# ---------------------- Data Analysis and Visualisation ----------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# ------------------- HARTH Cohort and Evaluation Utilities -------------------
from harth_data import (
    HARTH_SUBJECTS,
    ACTIVITY_NAMES,
    prepare_harth_feature_cohort,
)

from core.data_splitting import audit_group_class_coverage

from core.group_cross_validation import (
    evaluate_group_cross_validation,
    compare_models_by_subject,
)

# --------------------- Plot Style and Environment Check ----------------------
from core.plot_style import plot_style
plot_style()

print("HARTH full-development CV components: ready")


HARTH full-development CV components: ready


**02. Reconstruct the Complete 29-Person Development Cohort**

Reuse the source cohort and windowing settings from Notebook 08. Filter the eight-class task after constructing time-safe windows; retrieve cached recordings or download missing files as needed.

In [ ]:
# ============== 02. Reconstruct Full Development Feature Cohort ==============

# ------------------- Reserved Test and Development Cohorts -------------------
RESERVED_TEST = {"S023", "S032"}

DEVELOPMENT_SUBJECTS = sorted(set(HARTH_SUBJECTS) - RESERVED_TEST)

# -------------------- Eight-Class Task and Test Exclusion --------------------
TASK_CLASSES = tuple(range(1, 9))

assert len(DEVELOPMENT_SUBJECTS) == 29
assert not (set(DEVELOPMENT_SUBJECTS) & RESERVED_TEST)

# ------------------- Development Window and Feature Cohort -------------------
features, metadata, overview, excluded = prepare_harth_feature_cohort(
    subjects=DEVELOPMENT_SUBJECTS,
    data_dir=root / "data/raw/harth_v2",
    task_classes=TASK_CLASSES,  # Use the predefined documented activity classes.
    window_size=100, step_size=50, sampling_rate=50,
)

# ------------------------- Source Feature Alignment --------------------------
assert len(features) == len(metadata)

# ------------------------ Cohort Scope and Exclusions ------------------------
print("Development feature matrix:", features.shape)

print("Excluded out-of-task windows:", int(overview["excluded_windows"].sum()))
print("Reserved Test participants loaded: NO")


**03. Verify Subject-Disjoint Folds and Class Coverage**

Check that Train and Validation have no participant overlap and that the predefined classes are represented in each fold. The saved fold audit reports complete class coverage, but this alone does not establish model performance.

In [ ]:
# =============== 03. Audit All Five Participant-Disjoint Folds ===============

# -------------------- Participant-Disjoint Fold Coverage ---------------------
coverage, fold_audit = audit_group_class_coverage(
    labels=metadata["label"].to_numpy(),
    group_ids=metadata["subject_id"].to_numpy(),
    classes=TASK_CLASSES, n_splits=5,
)

# --------------------- Fold Membership and Class Support ---------------------
display(fold_audit[[
    "fold", "train_subjects", "validation_subjects",
    "train_windows", "validation_windows",
    "missing_train_classes", "missing_validation_classes",
]])

# --------------------------- Class Presence Checks ---------------------------
assert not fold_audit["missing_train_classes"].map(bool).any()

assert not fold_audit["missing_validation_classes"].map(bool).any()

print("Fold class-coverage audit: PASSED")


**04. Compare Three Models Across Five Development Folds**

Refit each classifier within every fold. Logistic Regression scaling uses that fold's Train observations only. Reserved Test recordings are not loaded.

In [ ]:
# =============== 04. Evaluate Classical Models with GroupKFold ===============

# ------------------ Five-Fold Development Model Evaluation -------------------
fold_results, subject_results, summary = evaluate_group_cross_validation(
    features=features.to_numpy(),
    labels=metadata["label"].to_numpy(),
    group_ids=metadata["subject_id"].to_numpy(),
    classes=TASK_CLASSES,
    n_splits=5, random_state=42,
)

# ---------------------- Model-Level Cross-Fold Summary -----------------------
display(summary.round(4))

# --------------------------- Held-Out Fold Details ---------------------------
display(fold_results[[
    "fold", "model", "validation_subjects",
    "validation_windows", "validation_classes_present",
    "macro_f1", "balanced_accuracy",
]].round(4))


**05. Visualise Variation in Held-Out Macro F1**

Compare fold-level Macro F1 across models, accounting for differences in participant composition and window counts. These folds share the Development cohort and do not represent independent replications.

In [ ]:
# ================= 05. Visualise Five-Fold Model Performance =================

# ---------------------- Fold-Level Macro F1 Comparison -----------------------
fig, ax = plt.subplots(figsize=(10, 5))

for model_name, table in fold_results.groupby("model", sort=False):
    table = table.sort_values("fold")
    ax.plot(table["fold"], table["macro_f1"], marker="o",
            label=f"{model_name} (mean={table['macro_f1'].mean():.3f})")

ax.set(title="HARTH: 29-person development grouped CV",
       xlabel="Held-out fold", ylabel="Macro F1",
       ylim=(0, 1), xticks=[1, 2, 3, 4, 5])
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()
plt.show()


**06. Compare Random Forest and Logistic Regression by Person**

Pair fixed eight-class Macro F1 scores for the same 29 held-out Development participants. The metric `difference_a_minus_b` is Random Forest minus Logistic Regression; positive values favour Random Forest.

In [ ]:
# ============== 06. Pair Random Forest and Logistic Regression ===============

# -------------------- Paired Participant Model Comparison --------------------
paired, paired_summary = compare_models_by_subject(
    subject_results,
    model_a="Random Forest",
    model_b="Logistic Regression",
    score_column="macro_f1_fixed_task",  # Use the same eight-class metric for both models.
)

# --------------------------- Paired Model Results ----------------------------
display(paired.round(4))

display(pd.DataFrame([paired_summary]))

# ------------------------ Signed Macro F1 Differences ------------------------
fig, ax = plt.subplots(figsize=(13, 5))

bar_colours = ["tab:blue" if value >= 0 else "tab:orange"
               for value in paired["difference_a_minus_b"]]

# -------------------- Participant-Level Difference Figure --------------------
ax.bar(paired["subject_id"], paired["difference_a_minus_b"], color=bar_colours)
ax.axhline(0, color="black", linewidth=0.8)
ax.set(title="Per-subject Macro F1 difference: Random Forest minus Logistic Regression",
       xlabel="Held-out development participant", ylabel="Paired Macro F1 difference")
ax.tick_params(axis="x", rotation=70)
ax.grid(axis="y", alpha=0.2)
fig.tight_layout()
plt.show()


**07. Compare Activity Support and Individual Model Scores**

Report class availability and fixed-task Macro F1 for each participant. Interpret individual differences alongside activity support rather than relying only on aggregate scores.

In [ ]:
# =============== 07. Compare Source Class Support and Results ================

# --------------------- Activity Class Support by Person ----------------------
person_support = pd.DataFrame({
    "subject_id": coverage.index,
    "eligible_windows": coverage.sum(axis=1).to_numpy(),
    "observed_classes": (coverage > 0).sum(axis=1).to_numpy(),
})

# ------------------------ Fixed-Task Scores by Model -------------------------
comparison = subject_results.pivot(
    index="subject_id", columns="model", values="macro_f1_fixed_task"
).reset_index()

# ---------------------- Participant Evidence Alignment -----------------------
participant_table = person_support.merge(comparison, on="subject_id", how="left")

# ---------------------------- Participant Summary ----------------------------
display(participant_table.round(4))

print("Independent development subjects represented:", len(participant_table))


**08. Scientific Quality Gate and Evidence Boundaries**

Verify the five-fold results, participant separation, source integrity and reserved Test exclusion. Because some properties of S023 and S032 were inspected in earlier exploratory work, do not describe the broader research as strictly preregistered and blind.

In [ ]:
# ============ 08. Development GroupKFold Scientific Quality Gate =============

# ------------------- Development Cohort and Test Exclusion -------------------
assert len(DEVELOPMENT_SUBJECTS) == len(set(DEVELOPMENT_SUBJECTS)) == 29

assert not (set(DEVELOPMENT_SUBJECTS) & RESERVED_TEST)

# ------------------ Participant and Fold Evidence Coverage -------------------
assert len(subject_results["subject_id"].unique()) == 29

assert not (set(subject_results["subject_id"]) & RESERVED_TEST)

assert len(fold_results) == 5 * 3

assert len(paired) == 29

# ------------------ Unique Evaluation and Feature Integrity ------------------
assert subject_results.groupby(["model", "subject_id"]).size().eq(1).all()

assert np.isfinite(features.to_numpy()).all()

# -------------------- Quality Gate and Evidence Boundary ---------------------
print("Full-development GroupKFold: PASSED")

print("Original final Test loaded or scored: NO")

print("Final independent benchmark claim: NOT SUPPORTED")
